# Validate Raw Meteorological Data

Compare ERA5 (low-res input) and CONUS404 (high-res target) fields to verify
data quality and spatial/temporal alignment before training.

**Usage:** Set `CASE_STUDY` below to point at your case study directory.

In [ ]:
import sys
from pathlib import Path

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import yaml

sys.path.insert(0, str(Path('..').resolve()))
from cosmos_wind_cnn.utils.config import load_config, parse_variable_config

%matplotlib inline

# --- Configuration ---
CASE_STUDY = '../case_studies/sf_bay'

case_dir = Path(CASE_STUDY)
processed_dir = case_dir / 'data' / 'processed'

train_config = load_config(case_dir / 'configs' / 'training.yaml')
input_vars, output_vars, wind_pair_indices = parse_variable_config(train_config)

print(f'Case study: {case_dir.name}')
print(f'Input vars: {input_vars}')
print(f'Output vars: {output_vars}')
print(f'Wind pairs: {wind_pair_indices}')

## 1. Load processed training data

In [ ]:
train_path = processed_dir / 'train.nc'
val_path = processed_dir / 'val.nc'

if not train_path.exists():
    raise FileNotFoundError(
        f'{train_path} not found. Run scripts/preprocess.py first.'
    )

ds_train = xr.open_dataset(train_path)
ds_val = xr.open_dataset(val_path)

print(f'Train: {len(ds_train.time)} timesteps')
print(f'Val:   {len(ds_val.time)} timesteps')
print(f'Variables: {list(ds_train.data_vars)}')

## 2. ERA5 vs CONUS404 distributions

Compare distributions of each input/output variable pair.

In [ ]:
pairs = train_config['variable_pairs']
n_pairs = len(pairs)
fig, axes = plt.subplots(n_pairs, 1, figsize=(12, 3.5 * n_pairs))
if n_pairs == 1:
    axes = [axes]

for ax, (pair_name, pair) in zip(axes, pairs.items()):
    low_var = pair['low_res']
    high_var = pair['high_res']

    low_data = ds_train[low_var].values.flatten()
    high_data = ds_train[high_var].values.flatten()

    # Subsample for faster plotting
    n_sample = min(500_000, len(low_data))
    idx = np.random.choice(len(low_data), n_sample, replace=False)

    ax.hist(low_data[idx], bins=80, alpha=0.6, label=f'{low_var} (ERA5)', density=True)
    ax.hist(high_data[idx], bins=80, alpha=0.6, label=f'{high_var} (CONUS404)', density=True)
    ax.set_title(f'{pair_name} Distribution', fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Temporal correlation

Time series at the grid center comparing low-res and high-res.

In [ ]:
# Select center point
cy = ds_train.dims.get('latitude', ds_train.dims.get('y', 0)) // 2
cx = ds_train.dims.get('longitude', ds_train.dims.get('x', 0)) // 2

first_pair_name = next(iter(pairs))
low_var = pairs[first_pair_name]['low_res']
high_var = pairs[first_pair_name]['high_res']

# Get spatial dim names
spatial_dims = [d for d in ds_train[low_var].dims if d != 'time']

ts_low = ds_train[low_var].isel({spatial_dims[0]: cy, spatial_dims[1]: cx})
ts_high = ds_train[high_var].isel({spatial_dims[0]: cy, spatial_dims[1]: cx})

# Plot first 500 timesteps for readability
n_show = min(500, len(ts_low))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts_low.time[:n_show], ts_low[:n_show], label=f'{low_var} (ERA5)', alpha=0.8)
ax.plot(ts_high.time[:n_show], ts_high[:n_show], label=f'{high_var} (CONUS404)', alpha=0.8)
ax.set_title(f'{first_pair_name} Time Series at Grid Center')
ax.set_xlabel('Time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Correlation
corr = np.corrcoef(ts_low.values, ts_high.values)[0, 1]
print(f'Pearson correlation ({low_var} vs {high_var}): {corr:.4f}')

## 4. Spatial difference maps

Mean difference between ERA5 and CONUS404 fields (bias).

In [ ]:
n_cols = min(3, n_pairs)
n_rows = (n_pairs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = np.atleast_2d(axes)

for i, (pair_name, pair) in enumerate(pairs.items()):
    row, col = divmod(i, n_cols)
    ax = axes[row, col]

    low_mean = ds_train[pair['low_res']].mean(dim='time')
    high_mean = ds_train[pair['high_res']].mean(dim='time')
    diff = high_mean - low_mean

    vmax = float(np.abs(diff).max())
    diff.plot(ax=ax, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'{pair_name}\nCONUS404 - ERA5 (mean)', fontsize=10)
    ax.set_aspect('equal')

# Hide unused axes
for j in range(i + 1, n_rows * n_cols):
    row, col = divmod(j, n_cols)
    axes[row, col].axis('off')

plt.suptitle('Mean Spatial Bias (CONUS404 - ERA5)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Wind speed and direction check

Verify that wind U and V components produce reasonable speed and direction fields.

In [ ]:
if wind_pair_indices:
    # Use the first wind pair
    wind_u_key = list(pairs.keys())[wind_pair_indices[0][0]]
    wind_v_key = list(pairs.keys())[wind_pair_indices[0][1]]

    u_high = pairs[wind_u_key]['high_res']
    v_high = pairs[wind_v_key]['high_res']
    u_low = pairs[wind_u_key]['low_res']
    v_low = pairs[wind_v_key]['low_res']

    # Compute wind speed for a single timestep
    t_idx = 100

    speed_low = np.sqrt(
        ds_train[u_low].isel(time=t_idx).values ** 2 +
        ds_train[v_low].isel(time=t_idx).values ** 2
    )
    speed_high = np.sqrt(
        ds_train[u_high].isel(time=t_idx).values ** 2 +
        ds_train[v_high].isel(time=t_idx).values ** 2
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    vmax = max(speed_low.max(), speed_high.max())

    im0 = axes[0].imshow(speed_low, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
    axes[0].set_title('ERA5 Wind Speed')
    plt.colorbar(im0, ax=axes[0], label='m/s')

    im1 = axes[1].imshow(speed_high, cmap='viridis', vmin=0, vmax=vmax, origin='lower')
    axes[1].set_title('CONUS404 Wind Speed')
    plt.colorbar(im1, ax=axes[1], label='m/s')

    fig.suptitle(f'Wind Speed at timestep {t_idx}', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('No wind pairs configured.')

## 6. Normalization statistics

In [ ]:
import pickle

stats_path = processed_dir / 'normalization_stats.pkl'
if stats_path.exists():
    with open(stats_path, 'rb') as f:
        stats = pickle.load(f)

    print(f'{"Variable":<25} {"Mean":>10} {"Std":>10} {"Min":>10} {"Max":>10}')
    print('-' * 65)
    for var, s in stats.items():
        print(f'{var:<25} {s["mean"]:>10.3f} {s["std"]:>10.3f} '
              f'{s["min"]:>10.3f} {s["max"]:>10.3f}')
else:
    print('Normalization stats not found. Run preprocessing first.')

ds_train.close()
ds_val.close()